# In-Class Activity: Prompt Engineering for Construction Documents
In this activity, we will use generative AI (GPT/HokieAI) to classify construction incident reports and review submittals against a specification, and compare how the wording of the prompt changes the accuracy of its outputs against the ground truth.

In [ ]:
#@title ▶ Step 0 · Run me first (1 minute) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line.
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp7_prompt_engineering", "aec_prompt"
FOLDERS = ["mp7_prompt_engineering"]          # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_prompt import lab
lab.setup()


## Part 1 · Classifying Incident Reports
OSHA records every severe work-related injury reported by employers, with a short narrative and a code for the event that caused it. Here, you will use HokieAI to classify 30 construction incident reports into OSHA's construction Focus Four hazards (fall, struck-by, caught-in/between, electrocution) or other, and compare its output with OSHA's own classification.

In every HokieAI step, the notebook gives you a prompt; you send it to HokieAI (https://hokie.ai.vt.edu/), paste HokieAI's output back, and the notebook scores it against the ground truth (the correct output, known for every example). Each prompt version changes one thing, and the comparison step after each part charts every scored output by prompt version.

In [ ]:
#@title ▶ Step 1a · Browse the Reports { display-mode: "form" }
#@markdown Run the cell to browse the incident reports HokieAI will classify. OSHA's classification of each report is the ground truth; it stays hidden until you score an output.
rows = 10 #@param [5, 10, 30] {type:"raw"}
lab.show_reports(rows)


In [ ]:
#@title ▶ Step 1b · Classify with HokieAI { display-mode: "form" }
#@markdown Choose a prompt version, run the cell, and follow the instructions. Score every version, each in a new chat.
#@markdown Then send one version in a second new chat and score that output too: a generative AI model can give a different output to the same prompt.
#@markdown v1 · task only: the categories and the output format.
#@markdown v2 · role and definitions: adds a role and OSHA's definition of each category.
#@markdown v3 · definitions and worked examples: adds five reports with OSHA's classification (few-shot prompting).
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
#@markdown Confusion matrix: a table that counts, for each of OSHA's classes (rows), the classes in HokieAI's output (columns); the diagonal holds the reports classified as OSHA classified them.
#@markdown If HokieAI declines the prompt or its output is blocked, start a new chat and send the prompt again.
prompt_version = "v1 · task only" #@param ["v1 · task only", "v2 · role and definitions", "v3 · definitions and worked examples", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """Classify each construction incident report below into one category: Fall, Struck-by, Caught-in/between, Electrocution, or Other.
Reply with one line per report and nothing else, in this format:
N01, Fall
N02, Other

Incident reports:
{reports}"""
v2_prompt_text = """You are a construction safety analyst coding OSHA severe-injury reports.
Definitions (OSHA construction Focus Four, as coded in OSHA's injury records):
- Fall: the worker fell to a lower level (from a ladder, roof, scaffold, through an opening), fell on the same level, or jumped down.
- Struck-by: the worker was hit by a moving object or equipment: a falling, flying, swinging, or rolling object, a vehicle, or a tool or object that slipped or swung while the worker was holding it (for example, a saw kickback or a slipping knife).
- Caught-in/between: the worker or a body part was caught in, squeezed, compressed, pinched, or crushed between objects or in running machinery, including trench, structure, or material collapses.
- Electrocution: the worker made contact with electrical energy (power lines, energized wiring or equipment).
- Other: anything else (heat, chemicals, fire or explosion, overexertion, struck against a stationary object, slips or trips without a fall).

Classify each construction incident report below into one category: Fall, Struck-by, Caught-in/between, Electrocution, or Other.
Reply with one line per report and nothing else, in this format:
N01, Fall
N02, Other

Incident reports:
{reports}"""
v3_prompt_text = """You are a construction safety analyst coding OSHA severe-injury reports.
Definitions (OSHA construction Focus Four, as coded in OSHA's injury records):
- Fall: the worker fell to a lower level (from a ladder, roof, scaffold, through an opening), fell on the same level, or jumped down.
- Struck-by: the worker was hit by a moving object or equipment: a falling, flying, swinging, or rolling object, a vehicle, or a tool or object that slipped or swung while the worker was holding it (for example, a saw kickback or a slipping knife).
- Caught-in/between: the worker or a body part was caught in, squeezed, compressed, pinched, or crushed between objects or in running machinery, including trench, structure, or material collapses.
- Electrocution: the worker made contact with electrical energy (power lines, energized wiring or equipment).
- Other: anything else (heat, chemicals, fire or explosion, overexertion, struck against a stationary object, slips or trips without a fall).

Worked examples:
{examples}

Classify each construction incident report below into one category: Fall, Struck-by, Caught-in/between, Electrocution, or Other.
Reply with one line per report and nothing else, in this format:
N01, Fall
N02, Other

Incident reports:
{reports}"""
steps_text = """1. Copy the prompt below.
2. Open {url} (sign in with your VT account), start a new chat, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
accuracy_text = """**Accuracy ({version}):** {right} of {n} reports classified as OSHA classified them (**{accuracy} %**)."""
matrix_text = """Confusion matrix (rows: OSHA's classification; columns: HokieAI's output):"""
disagree_text = """Reports where HokieAI's output differs from OSHA's classification:"""
columns_text = """Report | Report text | OSHA's classification | HokieAI's output"""
missing_text = """HokieAI's output has {got} of the {n} reports; the missing ones count as wrong."""
lab.classify_incidents(prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, accuracy_text=accuracy_text, matrix_text=matrix_text, disagree_text=disagree_text, columns_text=columns_text, missing_text=missing_text)


In [ ]:
#@title ▶ Step 1c · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after scoring the prompt versions in Step 1b. Each bar is the share of reports classified as OSHA classified them, over all outputs of that version.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("incidents", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 1
> From Step 1b: Report the accuracy of each prompt version and include the confusion matrix of the best one. Did the second chat give the same output?

> From Step 1c: Which change to the prompt (role, definitions, worked examples) improved the accuracy most?

> Choose two reports where HokieAI's output differs from OSHA's classification. From the report text, explain why each is hard to classify, and how you would change the prompt to fix it. Would you use HokieAI to classify a company's incident reports, and what would you check first?

## Part 2 · Reviewing Submittals Against the Specification
Before a material is used, its submittal (here, a concrete mix design or a mortar and grout mix) is checked against the specification. Twelve practice submittals were made for this course, each to be checked against a specification excerpt adapted from the Unified Facilities Guide Specifications (UFGS 03 30 00 Cast-in-Place Concrete and 04 20 00 Masonry). A submittal states its intended use, not the portion of the structure named in the specification, so the applicable requirements must be identified first. Some values must be computed from the batch weights, some are exactly at a limit, and two submittals comply fully. The ground truth is the list of planted noncompliances. Every requirement the submittal is checked against counts as one decision (six per submittal); a flag on a requirement that sets no limit for the intended use is a false flag.

In [ ]:
#@title ▶ Step 2a · Review a Submittal with HokieAI { display-mode: "form" }
#@markdown Choose a submittal and a prompt version, run the cell, and follow the instructions. Run all three versions on at least two submittals, including a concrete and a masonry one.
#@markdown Noncompliance: a submitted value that does not meet a requirement of the specification. False flag: a requirement HokieAI marks as noncompliant that complies.
#@markdown v1 · task only: the specification, the submittal, and the request to list every noncompliance.
#@markdown v2 · requirements first, then compare: asks HokieAI to list the applicable requirements first and then check them one by one (task decomposition).
#@markdown v3 · comparison table: adds a fixed comparison table and asks HokieAI to compute values the submittal does not state, such as w/cm.
#@markdown my own prompt: starts from v3; edit the prompt in the prompt box before copying it.
#@markdown Each review needs a new chat, so that HokieAI does not carry over an earlier review.
submittal = "Concrete mix design 1 · basement walls" #@param ["Concrete mix design 1 · basement walls", "Concrete mix design 2 · warehouse floor slab", "Concrete mix design 3 · entrance stoop and sidewalks", "Concrete mix design 4 · spread footings", "Concrete mix design 5 · stair-tower foundation walls", "Concrete mix design 6 · loading-dock apron", "Mortar and grout 1 · maintenance shop walls", "Mortar and grout 2 · office partitions", "Mortar and grout 3 · elevator pit walls", "Mortar and grout 4 · brick veneer backup walls", "Mortar and grout 5 · storage room partitions", "Mortar and grout 6 · loading-dock retaining wall"]
prompt_version = "v1 · task only" #@param ["v1 · task only", "v2 · requirements first, then compare", "v3 · comparison table", "my own prompt"]
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
v1_prompt_text = """Review the submittal below against the specification excerpt. List every noncompliance.
{final}

Specification:
{spec}
Submittal:
{submittal}"""
v2_prompt_text = """You are a project engineer reviewing a submittal. First list every requirement in the specification excerpt that applies to the submittal's intended use. Then compare the submittal with each requirement, one by one, and decide whether it complies.
{final}

Specification:
{spec}
Submittal:
{submittal}"""
v3_prompt_text = """You are a project engineer reviewing a submittal. First list every requirement in the specification excerpt that applies to the submittal's intended use. Then compare the submittal with each requirement, one by one, in a table with the columns: requirement | required value | submitted value | complies (YES or NO). Compute any value the submittal does not state directly (for example, w/cm = water / total cementitious materials).
{final}

Specification:
{spec}
Submittal:
{submittal}"""
final_text = """End your reply with one line per noncompliance, in this format:
NONCOMPLIANT | <requirement> | <submitted value> | <required value>
If the submittal complies with every requirement, end with: NONCOMPLIANT | none"""
steps_text = """1. Copy the prompt below.
2. Open {url} (sign in with your VT account), start a new chat, and paste the prompt.
3. Copy HokieAI's output and paste it into the box below.
4. Click *Score HokieAI output* to compare it with the ground truth."""
button_text = """Score HokieAI output"""
reply_text = """Paste HokieAI's full output here."""
prompt_label_text = """prompt"""
reply_label_text = """output"""
result_text = """**{version}, {submittal}**
**Requirements judged correctly:** {right} of {n}
**Noncompliances found:** {found} of {planted}
**False flags:** {false}"""
table_text = """Ground truth vs. HokieAI's output (each requirement the submittal is checked against):"""
columns_text = """Requirement | Ground truth | HokieAI's output | Correct"""
lab.review_submittal(submittal, prompt_version, v1_prompt_text=v1_prompt_text, v2_prompt_text=v2_prompt_text, v3_prompt_text=v3_prompt_text, final_text=final_text, steps_text=steps_text, button_text=button_text, reply_text=reply_text, prompt_label_text=prompt_label_text, reply_label_text=reply_label_text, result_text=result_text, table_text=table_text, columns_text=columns_text)


In [ ]:
#@title ▶ Step 2b · Compare Prompt Versions { display-mode: "form" }
#@markdown Run the cell after Step 2a. Each bar is the share of requirements judged correctly (complies or noncompliant), over all submittals scored with that version.
# The wording the notebook shows (prompts, instructions, results): edit the text between the triple quotes.
# Words in {braces} are filled in by the notebook; **bold** and *italic* work; each line is shown as its own line.
title_text = """Share of correct outputs by prompt version"""
table_text = """All scored outputs so far:"""
empty_text = """No scored outputs yet: run the step above first."""
axis_text = """correct (%)"""
lab.compare("submittals", title_text=title_text, table_text=table_text, empty_text=empty_text, axis_text=axis_text)


### 📝 Report question 2
> From Step 2a: Report, for each prompt version, the requirements judged correctly, the noncompliances found, and the false flags. Include one table.

> From Step 2b: Which prompt version judged the most requirements correctly? Why does asking for the applicable requirements first change the result?

> For one missed noncompliance or false flag, explain from HokieAI's output why it happened: a misread requirement, a value that had to be computed, or a requirement that does not apply to the intended use. On a real project, which is more costly, a missed noncompliance or a false flag, and why?

### Data and model sources
- Incident reports: Severe Injury Reports (2015–2025), Occupational Safety and Health Administration, U.S. Department of Labor, public domain, https://www.osha.gov/severe-injury-reports.
- Specification excerpts: adapted from the Unified Facilities Guide Specifications 03 30 00 and 04 20 00, U.S. Department of Defense, public domain, https://www.wbdg.org/dod/ufgs; the submittals are practice submittals made for CEM4644.
- Generative AI: GPT models via HokieAI (Virginia Tech).